In [39]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
# import torchvision.transforms.functional as TF
import torch.nn.functional as F

from PIL import Image
from pathlib import Path

import numpy as np
import pandas as pd
import pickle

import matplotlib.pyplot as plt

# ML model to estimate the parameter values

* G_align
* G_density
* G_curve
* G_conn
* L_align
* L_density
* L_conn
* L_curve
* spline_length
* spline_num
* wave_amplitude_px
* wave_wavelength_px (can be None)
* L_wave_freq

## Input: pickle files

* 256x256 image np.arrays, normalized [0,1]
* metadata 


* Run filtration??

In [ ]:
class ImageDataset(Dataset):
    def __init__(self, img_path):
        self.pkl_paths = list(Path(img_path).glob("*pkl"))

    def __len(self):
        return len(self.pkl_path)
    
    def __getitem__(self, idx):
        # Get path
        pkl_path = self.pkl_paths[idx]

        # Load file
        with open('pickle_path', 'rb') as file:
            data = pickle.load(file)

        img = torch.tensor(data['img']) # Is a np.array
        density = torch.tensor(data['density'])
        vector = torch.tensor(data['vector'])

        meta_dict = data['meta'] # Dict of metadata
        
        return img, meta_dict, str(img_path)


def load_metadata(meta_dict):
    G_align = meta_dict.get("G_align")
    G_density = meta_dict.get("G_density")
    G_curve = meta_dict.get("G_curve")
    G_conn = meta_dict.get("G_conn")

    L_align = meta_dict.get("L_align")
    L_density = meta_dict.get("L_density")
    L_conn = meta_dict.get("L_conn")
    L_curve = meta_dict.get("L_curve")

    spline_length = meta_dict.get("spline_length")
    spline_num = meta_dict.get("spline_num")

    wave_amplitude_px = meta_dict.get("wave_amplitude_px")
    wave_wavelength_px = meta_dict.get("wave_wavelength_px")  # Can be None
    L_wave_freq = meta_dict.get("L_wave_freq")

    return (
        G_align,
        G_density,
        G_curve,
        G_conn,
        L_align,
        L_density,
        L_conn,
        L_curve,
        spline_length,
        spline_num,
        wave_amplitude_px,
        wave_wavelength_px,
        L_wave_freq,
    )


In [ ]:
res = syn.generate_synthetic_shg(
    seed=10, G_align=0.3, G_density=0.7, G_curve=0, G_conn=0.9,
    L_align=0.4, L_density=0.4, L_conn=1, L_curve=0.6,
    spline_length=60, spline_num=1000,
    wave_amplitude_px=6,      # bump for visibly wavier fibers
    wave_wavelength_px=30,    # None -> auto from fiber_width_px
    L_wave_freq=0.5,
    show_plots=True,
    minimal=True,
)

In [ ]:
{
    'seed': ,
    'G_align': ,
    'G_density': ,
    'G_curve': ,
    'G_conn': ,
    'L_align': ,
    'L_density':,
    'L_conn': ,
    'L_curve': ,
    'spline_length': ,
    'spline_num': ,
    'wave_amplitude_px': ,
    'wave_wavelength_px': ,
    'L_wave_freq': ,
}

In [ ]:
class SyntheticCNN(nn.Module):
    def __init__(
        self, 
        n_channels=5, 
        n_filters=16
    ):
    super(SyntheticCNN, self).__init__()
    self.encoder = nn.Sequential(
        nn.Conv2d(n_channels, n_filters, padding=1),
        nn.GELU(),
        nn.MaxPool2d(kernel_size=2, stride=2),

        nn.Conv2d(n_filters, n_filters*2, padding=1),
        nn.GELU(),
        nn.MaxPool2d(kernel_size=2, stride=2),

        nn.Conv2d(n_filters*2, n_filters*4, padding=1),
        nn.GELU(),
        nn.MaxPool2d(kernel_size=2, stride=2),
    )

    # self.fcl = nn.linear() #nn.Flatten()

    self.decoder = nn.Sequential(
        nn.ConvTranspose2d(n_filters*4, n_filters*2, stride=2, padding=1, output_padding=1),
        nn.GELU(),
        nn.Conv




    )

class Decoder

In [44]:
class EncoderBlock(nn.Module):
    def __init__(self, in_channels, num_filters):
        super(EncoderBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, num_filters, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(num_filters, num_filters, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d((2, 2), stride=2)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        skip_features = F.relu(self.conv2(x)) # We want this for the skip connection
        pool_out = self.pool(skip_features)   # This goes to the next encoder layer
        return skip_features, pool_out
    

class DecoderBlock(nn.Module):
    def __init__(self, in_channels, num_filters, skip_channels):
        super(DecoderBlock, self).__init__()

        self.convT = nn.ConvTranspose2d(in_channels, num_filters, kernel_size=2, stride=2)
        self.conv1 = nn.Conv2d(num_filters + skip_channels, num_filters, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(num_filters, num_filters, kernel_size=3, padding=1)

    def forward(self, x, skip_features):
        x = self.convT(x)
        # Skip features
        if skip_features.shape[2:] != x.shape[2:]:
            skip_features = TF.resize(skip_features, size=x.shape[2:])
        
        # Concate skips
        x = torch.cat([x, skip_features], dim=1)

        # ReLU
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))

        return x


class UNET(nn.Module):
    def __init__(self, 
                 input_img_size=512,
                 num_filters=64, # for a 512 image
                 in_channels=1,  # n_channels, 3d will have more...
                 num_classes=1,   # grayscale images, numclasses=1
                 num_params=12):
        super(UNET, self).__init__()

        self.enc1 = EncoderBlock(in_channels, num_filters)
        self.enc2 = EncoderBlock(num_filters, num_filters*2)
        self.enc3 = EncoderBlock(num_filters*2, num_filters*4)
        self.enc4 = EncoderBlock(num_filters*4, num_filters*8)

        self.bottleneck_conv1 = nn.Conv2d(num_filters*8, num_filters*16, kernel_size=3, padding=1)
        self.bottleneck_conv2 = nn.Conv2d(num_filters*16, num_filters*16, kernel_size=3, padding=1)

        self.dec1 = DecoderBlock(num_filters*16, num_filters*8, num_filters*8) #input, skip, filters
        self.dec2 = DecoderBlock(num_filters*8, num_filters*4, num_filters*4)
        self.dec3 = DecoderBlock(num_filters*4, num_filters*2, num_filters*2)
        self.dec4 = DecoderBlock(num_filters*2, num_filters, num_filters)

        self.final_conv = nn.Conv2d(num_filters, num_classes, kernel_size=1) #Replace with the classifiers

        self.density_head = nn.Sequential(
            nn.Conv2d(num_filters, num_filters // 2, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(num_filters // 2, 1, kernel_size=1),
            nn.Sigmoid() #[0,1]
        )

        self.vector_head = nn.Sequential(
            nn.Conv2d(num_filters, num_filters // 2, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(num_filters // 2, 1, kernel_size=1),
            nn.Sigmoid() #[0,1], maybe tan
        )

        flat_size = (input_img_size // 16)**2 * (num_filters*16) #img_size * num_filters
        self.meta_head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(flat_size, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, num_params)
        )

    def forward(self, x):
        s1, p1 = self.enc1(x)
        s2, p2 = self.enc2(p1)
        s3, p3 = self.enc3(p2)
        s4, p4 = self.enc4(p3)

        b1 = self.bottleneck_conv1(p4)
        b2 = self.bottleneck_conv2(b1)

        meta = self.meta_head(b2)

        d1 = self.dec1(b2, s4)
        d2 = self.dec2(d1, s3)
        d3 = self.dec3(d2, s2)
        d4 = self.dec4(d3, s1)

        shg = self.final_conv(d4)
        density = self.density_head(d4)
        vector = self.vector_head(d4)

        return shg, meta, density, vector #self.final_conv(d4)


In [46]:
model = UNET(num_filters=64, in_channels=1)

x = torch.randn(2, 1, 512, 512)

shg, meta, density, vector = model(x)